# Likelihood fields and scan matching: notebook

The room of the beam-model Note (5 m x 4 m) with a box from x = 3.6 to 4.2 m and y = 0.6 to 1.2 m, on a grid of 1 cm cells. The robot stands at pose A = (2, 2, 0); a person 1 m ahead blocks its forward beam. We check the Note's numbers for the likelihood field, compare it with the beam model, and run scan matching and correlation-based map matching.

In [1]:
import time
import numpy as np
from scipy.ndimage import distance_transform_edt

RES = 0.01                                 # cell size (m)
X0, Y0 = -0.2, -0.2                        # world position of the grid's lower-left corner
NX, NY = 540, 440                         # grid covers x in [-0.2, 5.2), y in [-0.2, 4.2)
ZMAX = 5.0
SIG = 0.1                                  # spread of the endpoint Gaussian (m)
W_HIT, W_RAND = 0.9, 0.1                   # weights of the two parts of the likelihood field
ANGLES = np.radians(np.arange(0, 360, 30))
POSE_A = (2.0, 2.0, 0.0)
BOX = (3.6, 4.2, 0.6, 1.2)


def make_map(box=True):
    """Occupancy grid: True = occupied. Walls are the cells outside the 5 m x 4 m room."""
    xs = X0 + (np.arange(NX) + 0.5) * RES
    ys = Y0 + (np.arange(NY) + 0.5) * RES
    X, Y = np.meshgrid(xs, ys, indexing="xy")              # arrays of shape (NY, NX)
    occ = (X < 0) | (X > 5) | (Y < 0) | (Y > 4)
    if box:
        occ |= (X > BOX[0]) & (X < BOX[1]) & (Y > BOX[2]) & (Y < BOX[3])
    return occ, xs, ys


def cell(x, y):
    return int(np.floor((y - Y0) / RES)), int(np.floor((x - X0) / RES))


def distance_grid(occ):
    """Distance (m) from each cell centre to the nearest occupied cell centre: the Euclidean distance transform."""
    return distance_transform_edt(~occ) * RES


def field(dist, sig=SIG):
    """Likelihood field: density of a beam endpoint in each cell (per m)."""
    g = np.exp(-0.5 * (dist / sig) ** 2) / (sig * np.sqrt(2 * np.pi))
    return W_HIT * g + W_RAND / ZMAX


def raycast(occ, pose, ang, step=0.005):
    x, y, th = pose
    d = 0.0
    while d < ZMAX:
        r, c = cell(x + d * np.cos(th + ang), y + d * np.sin(th + ang))
        if r < 0 or c < 0 or r >= NY or c >= NX or occ[r, c]:
            return d
        d += step
    return ZMAX


def endpoints(pose, z, angles=ANGLES):
    x, y, th = pose
    z = np.asarray(z, float)
    keep = z < ZMAX
    return x + z[keep] * np.cos(th + angles[keep]), y + z[keep] * np.sin(th + angles[keep])


def lookup(grid, px, py):
    """Value of a grid at world points; points outside the grid get the far-away floor value."""
    out = np.full(len(px), W_RAND / ZMAX)
    for i, (a, b) in enumerate(zip(px, py)):
        r, c = cell(a, b)
        if 0 <= r < NY and 0 <= c < NX:
            out[i] = grid[r, c]
    return out


def field_loglik(F, pose, z, angles=ANGLES):
    px, py = endpoints(pose, z, angles)
    return np.log(lookup(F, px, py)).sum()


def scan_A(seed=1):
    """The 12 readings at pose A: true ranges plus small noise; the forward beam reads the person at 1.00 m and
    the four beams of the beam-model Note keep their readings 1.00, 1.97, 2.04, 1.99."""
    occ, _, _ = make_map()
    rng = np.random.default_rng(seed)
    z = np.array([raycast(occ, POSE_A, a) for a in ANGLES])
    z = np.round(z + rng.normal(0, 0.02, len(z)), 2)
    z[0], z[3], z[6], z[9] = 1.00, 1.97, 2.04, 1.99
    return z


# ---- scan matching: two scans of 360 beams, one every degree, no person ----
ANG72 = np.radians(np.arange(0, 360, 1))
POSE_1 = (2.0, 2.0, 0.0)
POSE_2 = (2.3, 2.1, np.radians(5))          # true pose of the second scan; odometry says (2.3, 2.0, 0)
DTH = np.radians(np.arange(-10, 10.5, 1))
DX = np.round(np.arange(-0.1, 0.7001, 0.02), 2)
DY = np.round(np.arange(-0.3, 0.3001, 0.02), 2)


def scan72(pose, seed):
    occ, _, _ = make_map()
    rng = np.random.default_rng(seed)
    z = np.array([raycast(occ, pose, a) for a in ANG72])
    return np.round(z + rng.normal(0, 0.02, len(z)), 2)


def scan_field(z1):
    """Likelihood field built from the endpoints of scan 1 (in scan 1's frame, which is the world frame here)."""
    occ = np.zeros((NY, NX), bool)
    px, py = endpoints((0.0, 0.0, 0.0), z1, ANG72)
    for a, b in zip(px + POSE_1[0], py + POSE_1[1]):
        r, c = cell(a, b)
        if 0 <= r < NY and 0 <= c < NX:
            occ[r, c] = True
    return field(distance_grid(occ))


def match(F1, z2):
    """Score every candidate motion (dx, dy, dth) of scan 2 relative to scan 1: sum of log field values."""
    qx, qy = endpoints((0.0, 0.0, 0.0), z2, ANG72)
    best, scores = None, np.empty((len(DTH), len(DY), len(DX)))
    for i, t in enumerate(DTH):
        rx = np.cos(t) * qx - np.sin(t) * qy
        ry = np.sin(t) * qx + np.cos(t) * qy
        for j, dy in enumerate(DY):
            for k, dx in enumerate(DX):
                scores[i, j, k] = np.log(lookup_fast(F1, rx + POSE_1[0] + dx, ry + POSE_1[1] + dy)).sum()
    i, j, k = np.unravel_index(scores.argmax(), scores.shape)
    return (DX[k], DY[j], np.degrees(DTH[i])), scores


def lookup_fast(grid, px, py):
    r = np.floor((py - Y0) / RES).astype(int)
    c = np.floor((px - X0) / RES).astype(int)
    ok = (r >= 0) & (r < NY) & (c >= 0) & (c < NX)
    out = np.full(len(px), W_RAND / ZMAX)
    out[ok] = grid[r[ok], c[ok]]
    return out


# ---- correlation-based map matching on 0.1 m cells ----
MRES = 0.1
MX = np.arange(0.05, 5.0, MRES)            # cell centres across the room (50 x 40 cells)
MY = np.arange(0.05, 4.0, MRES)


def global_map():
    """1 for cells that touch a surface (the walls' inner faces and the box), 0 for free cells."""
    g = np.zeros((len(MY), len(MX)))
    for j, y in enumerate(MY):
        for i, x in enumerate(MX):
            near_wall = x < MRES or x > 5 - MRES or y < MRES or y > 4 - MRES
            in_box = BOX[0] - MRES < x < BOX[1] + MRES and BOX[2] - MRES < y < BOX[3] + MRES
            inside_box = BOX[0] + MRES < x < BOX[1] - MRES and BOX[2] + MRES < y < BOX[3] - MRES
            g[j, i] = 1.0 if near_wall or (in_box and not inside_box) else 0.0
    return g


def local_map(z2, dx, dy, dth):
    """The scan's endpoints, placed at a candidate pose, marked on the same 0.1 m cells (ends outside are dropped)."""
    qx, qy = endpoints((0.0, 0.0, 0.0), z2, ANG72)
    px = np.cos(dth) * qx - np.sin(dth) * qy + POSE_1[0] + dx
    py = np.sin(dth) * qx + np.cos(dth) * qy + POSE_1[1] + dy
    l = np.zeros((len(MY), len(MX)))
    i = np.clip(np.floor(px / MRES).astype(int), 0, len(MX) - 1)
    j = np.clip(np.floor(py / MRES).astype(int), 0, len(MY) - 1)
    l[j, i] = 1.0
    return l


def corr(g, l):
    """Pearson correlation of two maps, cell by cell."""
    a, b = g.ravel() - g.mean(), l.ravel() - l.mean()
    return (a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum())


The beam model of the beam-model Note, for comparison:

In [2]:
from math import erf, sqrt

import numpy as np


W = dict(hit=0.75, short=0.12, max=0.05, rand=0.08)
SIG_HIT, LAM, STEP = 0.05, 2.0, 0.01


def mixture(z, zs):
    area = 0.5 * (erf((ZMAX - zs) / (SIG_HIT * sqrt(2))) - erf(-zs / (SIG_HIT * sqrt(2))))
    hit = np.exp(-0.5 * ((z - zs) / SIG_HIT) ** 2) / (SIG_HIT * sqrt(2 * np.pi)) / area if 0 <= z <= ZMAX else 0.0
    short = lam_short(z, zs)
    mx = 1 / STEP if z >= ZMAX - STEP / 2 else 0.0
    return W["hit"] * hit + W["short"] * short + W["max"] * mx + W["rand"] / ZMAX


def lam_short(z, zs):
    if 0 <= z <= zs:
        return LAM * np.exp(-LAM * z) / (1 - np.exp(-LAM * zs))
    return 0.0


def beam_loglik(occ, pose, z, angles=ANGLES):
    return sum(np.log(mixture(zz, raycast(occ, pose, a))) for zz, a in zip(z, angles))


## 1. The likelihood field at pose A (Note, Sections 3 and 4)

In [3]:
occ, xs, ys = make_map()
D = distance_grid(occ)          # Euclidean distance transform: metres to the nearest occupied cell
F = field(D)                    # likelihood field: 0.9 N(d; 0, 0.1^2) + 0.1 / 5
z = scan_A()
px, py = endpoints(POSE_A, z)
import pandas as pd
tab = pd.DataFrame(dict(angle=np.degrees(ANGLES).round(), reading=z, end_x=px.round(3), end_y=py.round(3),
                        distance=lookup(D, px, py).round(3), field=lookup(F, px, py).round(4)))
tab

,angle,reading,end_x,end_y,distance,field
0,0.0,1.00,3.000,2.000,1.008,0.0200
1,30.0,3.48,5.014,3.740,0.000,3.6105
2,60.0,2.32,3.160,4.009,0.000,3.6105
3,90.0,1.97,2.000,3.970,0.030,3.4525
4,120.0,2.33,0.835,4.018,0.000,3.6105
5,150.0,2.32,-0.009,3.160,0.000,3.6105
6,180.0,2.04,-0.040,2.000,0.000,3.6105
7,210.0,2.32,-0.009,0.840,0.000,3.6105
8,240.0,2.32,0.840,-0.009,0.000,3.6105
9,270.0,1.99,2.000,0.010,0.020,3.5394


In [4]:
print('peak of the field (d = 0):', field(0.0).round(4), ' floor far away:', W_RAND / ZMAX)
print('log-likelihood of the scan at A:', field_loglik(F, POSE_A, z).round(2))
for P in [(3, 2, 0), (2, 2.3, 0), (2.1, 2, 0), (2, 2, np.radians(5))]:
    print(P, field_loglik(F, P, z).round(2))

peak of the field (d = 0): 3.6105  floor far away: 0.02
log-likelihood of the scan at A: 10.15
(3, 2, 0) -15.75
(2, 2.3, 0) -19.29
(2.1, 2, 0) 8.91
(2, 2, np.float64(0.08726646259971647)) 8.49


## 2. Cost: beam model against likelihood field (Note, Section 2.1)

In [5]:
poses = [(2, y, 0) for y in np.arange(1.9, 2.1, 0.01)]
t = time.time(); [beam_loglik(occ, p, z) for p in poses]; tb = (time.time() - t) / len(poses)
t = time.time(); [field_loglik(F, p, z) for p in poses]; tf = (time.time() - t) / len(poses)
print(f'beam model {tb * 1000:.2f} ms per pose, likelihood field {tf * 1000:.3f} ms per pose: {tb / tf:.0f} times faster')
print('beam model log-likelihood at A:', round(beam_loglik(occ, POSE_A, z), 2))

beam model 11.31 ms per pose, likelihood field 0.045 ms per pose: 253 times faster
beam model log-likelihood at A: 15.93


## 3. A beam that slips past the box corner (Note, Section 2.2)

In [6]:
a, zr = np.radians(330), 1.86
for y in [1.50, 1.52, 1.53, 1.55]:
    zs = raycast(occ, (2, y, 0), a)
    ex, ey = endpoints((2, y, 0), np.array([zr]), np.array([a]))
    print(f'y = {y}: z* = {zs:.2f}  beam model {mixture(zr, zs):.4f}  field {lookup(F, ex, ey)[0]:.4f}')

y = 1.5: z* = 3.00  beam model 0.0218  field 3.3344
y = 1.52: z* = 3.04  beam model 0.0218  field 3.5394
y = 1.53: z* = 1.85  beam model 5.8816  field 3.5926
y = 1.55: z* = 1.85  beam model 5.8816  field 3.6105


## 4. The field sees through the box (Note, Section 4.4)

In [7]:
C = (3.9, 2.0, -np.pi / 2)
zs = raycast(occ, C, 0.0)
print('expected range from C:', round(zs, 2), ' beam model of 1.98 m:', mixture(1.98, zs))
print('field at the endpoint (3.9, 0.02):', lookup(F, np.array([3.9]), np.array([0.02]))[0].round(4))

expected range from C: 0.8  beam model of 1.98 m: 0.016
field at the endpoint (3.9, 0.02): 3.4525


## 5. Scan matching (Note, Section 5)

Scan 1 is taken at (2, 2, 0), scan 2 at the true pose (2.3, 2.1, 5°); odometry claims (2.3, 2.0, 0). Each scan has 360 beams.

In [8]:
z1, z2 = scan72(POSE_1, 11), scan72(POSE_2, 12)
F1 = scan_field(z1)
t = time.time()
best, S = match(F1, z2)
print('best motion (dx, dy, dtheta deg):', best, f' searched {S.size} candidates in {time.time() - t:.1f} s')
th = list(np.round(np.degrees(DTH)))
print('score of the odometry guess:', S[th.index(0), list(DY).index(0.0), list(DX).index(0.3)].round(1))
print('score of the best:', S.max().round(1))

best motion (dx, dy, dtheta deg): (np.float64(0.3), np.float64(0.1), np.float64(5.0))  searched 26691 candidates in 0.9 s
score of the odometry guess: 169.3
score of the best: 447.1


The same search with only 72 beams (one every 5 degrees): the scan points lie up to 0.3 m apart, so the field built from scan 1 has gaps.

In [9]:
ANG_FULL = ANG72
ANG72 = np.radians(np.arange(0, 360, 5))
z1s, z2s = scan72(POSE_1, 11), scan72(POSE_2, 12)
best5, _ = match(scan_field(z1s), z2s)
print('best with 72 beams:', best5)
ANG72 = ANG_FULL

best with 72 beams: (np.float64(0.28), np.float64(0.1), np.float64(7.0))


## 6. Correlation-based map matching (Note, Section 6)

In [10]:
g_strip = np.array([0, 0, 1, 0, 0, 0, 1, 0.])     # global map: walls in cells 3 and 7
l_strip = np.array([0, 1, 0, 0, 0, 1, 0, 0.])     # local map as first placed
for s in range(0, 3):
    l = np.roll(l_strip, s)
    print('shift', s, l.astype(int), 'sum of products', (g_strip * l).sum(), ' correlation', corr(g_strip, l).round(3))

shift 0 [0 1 0 0 0 1 0 0] sum of products 0.0  correlation -0.333
shift 1 [0 0 1 0 0 0 1 0] sum of products 2.0  correlation 1.0
shift 2 [0 0 0 1 0 0 0 1] sum of products 0.0  correlation -0.333


In [11]:
g = global_map()
l_best = local_map(z2, 0.3, 0.1, np.radians(5))
print('cells: global map', int(g.sum()), 'marked of', g.size, '; local map', int(l_best.sum()))
print('correlation at the true motion:', corr(g, l_best).round(3))
print('correlation at the odometry guess:', corr(g, local_map(z2, 0.3, 0.0, 0.0)).round(3))
t = time.time()
bestm = max(((corr(g, local_map(z2, dx, dy, t_)), dx, dy, round(np.degrees(t_))) for t_ in DTH for dy in DY for dx in DX))
print('best placement (r, dx, dy, dtheta):', (round(bestm[0], 3),) + bestm[1:], f'{time.time() - t:.1f} s')

cells: global map 224 marked of 2000 ; local map 175
correlation at the true motion: 0.872
correlation at the odometry guess: 0.617


best placement (r, dx, dy, dtheta): (np.float64(0.877), np.float64(0.3), np.float64(0.08), 6) 2.9 s


In [12]:
all_ones = np.ones_like(g)
rng = np.random.default_rng(0)
half = (rng.random(g.shape) < 0.5).astype(float)
print('sum of products: true placement', (g * l_best).sum(), ' every cell marked', (g * all_ones).sum(),
      ' half the cells at random', (g * half).sum())
print('correlation of the random half map:', corr(g, half).round(4))

sum of products: true placement 175.0  every cell marked 224.0  half the cells at random 113.0
correlation of the random half map: 0.0007


## 7. Try it: slide scan 2 over the field of scan 1

In [13]:
import plotly.graph_objects as go
qx, qy = endpoints((0.0, 0.0, 0.0), z2, ANG72)
fig = go.Figure(go.Heatmap(x=xs[::4], y=ys[::4], z=F1[::4, ::4], colorscale='Viridis', showscale=False))
cands = [(dx, 0.1, 5) for dx in np.round(np.arange(0.0, 0.61, 0.1), 1)]
for i, (dx, dy, d) in enumerate(cands):
    t_ = np.radians(d)
    fig.add_trace(go.Scatter(x=np.cos(t_) * qx - np.sin(t_) * qy + 2 + dx, y=np.sin(t_) * qx + np.cos(t_) * qy + 2 + dy,
                             mode='markers', marker=dict(color='red', size=4), visible=(i == 3)))
steps = [dict(method='update', args=[{'visible': [True] + [j == i for j in range(len(cands))]}], label=str(c[0]))
         for i, c in enumerate(cands)]
fig.update_layout(sliders=[dict(steps=steps, active=3, currentvalue=dict(prefix='dx (m), with dy = 0.1, dθ = 5°: '))],
                  width=700, height=600, yaxis=dict(scaleanchor='x'))
fig